In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import pandas as pd
import os

drive_data_folder = '/content/drive/MyDrive/Colab Notebooks/DDA4210/DataSet'

# --- New code added for verification ---
print("\n--- 路径验证开始 ---")
# 检查 Google Drive 是否已挂载
if not os.path.exists('/content/drive'):
    print(" Google Drive 似乎没有挂载。请运行 `from google.colab import drive; drive.mount('/content/drive')` 来挂载。")
elif not os.path.exists(drive_data_folder):
    print(f" 指定的数据文件夹不存在: {drive_data_folder}\n请检查路径是否正确，或者文件是否已上传到 Google Drive。")
else:
    print(f" Google Drive 已挂载。正在检查数据文件夹: {drive_data_folder}")
    print("以下是该文件夹中的内容：")
    try:
        for item in os.listdir(drive_data_folder):
            print(f"  - {item}")
    except Exception as e:
        print(f"   无法列出文件夹内容，可能存在权限问题或路径错误: {e}")
print("--- 路径验证结束 ---\n")
# --- End of new code ---


# 代码自动去匹配大小写
files_and_cols = {
    "DEMO_J": ['SEQN', 'RIAGENDR', 'RIDAGEYR', 'DMDEDUC2', 'INDFMPIR', 'DMDMARTL'],
    "DPQ_J": ['SEQN', 'DPQ010', 'DPQ020', 'DPQ030', 'DPQ040', 'DPQ050', 'DPQ060', 'DPQ070', 'DPQ080', 'DPQ090'],
    "SLQ_J": ['SEQN', 'SLD012'],
    "PAQ_J": ['SEQN', 'PAD680'],
    "BMX_J": ['SEQN', 'BMXBMI'],
    "MCQ_J": ['SEQN', 'MCQ220', 'MCQ160C'],
    "SMQ_J": ['SEQN', 'SMQ020'],
    "ALQ_J": ['SEQN', 'ALQ130']
}

df_final = None

print("开始从 Google Drive 读取数据...")
for file_base, cols in files_and_cols.items():
    # 自动探测是 .XPT 还是 .xpt
    file_path_upper = os.path.join(drive_data_folder, f"{file_base}.XPT")
    file_path_lower = os.path.join(drive_data_folder, f"{file_base}.xpt")

    if os.path.exists(file_path_upper):
        actual_path = file_path_upper
    elif os.path.exists(file_path_lower):
        actual_path = file_path_lower
    else:
        print(f" 找不到文件: {file_base} (检查是否上传或拼写错误)")
        continue

    print(f" 正在读取: {os.path.basename(actual_path)} ...")
    df_temp = pd.read_sas(actual_path, format='xport')[cols]

    if df_final is None:
        df_final = df_temp
    else:
        df_final = pd.merge(df_final, df_temp, on='SEQN', how='outer')

if df_final is not None:
    print("\n 所有数据合并完成！目前的数据维度是:", df_final.shape)
    save_path = os.path.join(drive_data_folder, 'NHANES_Merged_Data.csv')
    df_final.to_csv(save_path, index=False)
    print(f" 整合后的数据已永久保存至: {save_path}")
    display(df_final.head())
else:
    print(" 合并失败，因为一个文件都没读到！请务必检查 drive_data_folder 路径！")